# CONFIGURACIÓN DE SPARK 
<p>Dentro de este ejercicio se revisara la configuración básica de Spark y la diferencia con las bases de datos tradicionales </p>
<p>Se realizará principalmente la instalación, selección y agrupamiento de datos para posteriormente su ordenamiento</p>

-----------------------------------

### Configuración de Plataforma Spark

In [8]:
#Se importan librerias de spark para realizar el dataframe
from pyspark.sql import SparkSession


In [9]:
#Se crea cliente de spark
spark = SparkSession \
        .builder \
        .appName('LecturaArchivoCSV') \
        .getOrCreate()

### Importacion de datos

In [10]:
#Se importa el archivo a utilizar 
path_archivo = 'databses/kc_house_data.csv'
house = spark.read.csv(path_archivo, header= True)

In [11]:
#Se visualiza el dataframe creado
house.show()

+----------+---------------+----------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|     price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+----------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|    221900|       3|        1|       1180|    5650|     1|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
|6414100192|20141209T000000|    538000|       3|     2.25|       2570|    7242|     2|         0|   0|        3|    7|      2170|       

In [12]:
#Ubicar el tipo de dato que contiene el dataframe
house.dtypes

[('id', 'string'),
 ('date', 'string'),
 ('price', 'string'),
 ('bedrooms', 'string'),
 ('bathrooms', 'string'),
 ('sqft_living', 'string'),
 ('sqft_lot', 'string'),
 ('floors', 'string'),
 ('waterfront', 'string'),
 ('view', 'string'),
 ('condition', 'string'),
 ('grade', 'string'),
 ('sqft_above', 'string'),
 ('sqft_basement', 'string'),
 ('yr_built', 'string'),
 ('yr_renovated', 'string'),
 ('zipcode', 'string'),
 ('lat', 'string'),
 ('long', 'string'),
 ('sqft_living15', 'string'),
 ('sqft_lot15', 'string')]

In [ ]:
#Se cambia el tipo de dato para que no tenga errores
import pyspark.sql.functions as F
house = (house
    .withColumn('price',F.col('price').cast('double'))
    .withColumn('bedrooms',F.col('bedrooms').cast('double'))
    .withColumn('bathrooms',F.col('bathrooms').cast('double'))
    .withColumn('zipcode',F.col('zipcode').cast('int'))
    .withColumn('lat',F.col('lat').cast('double'))
    .withColumn('long',F.col('long').cast('double'))
)

### Selección de datos con filtros sencillos

In [14]:
#Listado completo de columnas ordenado por zipcode
import pyspark.sql.functions as F1
house.selectExpr('try_cast(zipcode as int)')
house.sort(F1.col('zipcode')).show(10)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|   price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|8159610030|20140722T000000|196000.0|     3.0|     2.25|       2070|   11576|     2|         0|   0|        3|    7|      2070|            0|    1974|           0|  98001|47.3417|-122.271|         1890|      7519|
|3751600030|20140717T000000|100000.0|     2.0|      1.0|        770|   17334|     1|         0|   0|        3|    7|       770|            0|   

-------------------

In [15]:
#Zipcode con mayor numero de casas, calculo de promedio de precio, tamaño en m2
house.createOrReplaceTempView('HOUSES')
#Se ejecuta la consulta tipo sql
sql_str = 'select zipcode, avg(price), lat * long as Metros_Cuadrados from HOUSES group by zipcode, price,Metros_Cuadrados'
spark.sql(sql_str).show()

+-------+----------+-------------------+
|zipcode|avg(price)|   Metros_Cuadrados|
+-------+----------+-------------------+
|  98133|  420000.0|      -5837.9045544|
|  98133|  371500.0| -5840.791459800001|
|  98045|  415000.0|      -5776.5987976|
|  98070|  369900.0|      -5811.9237154|
|  98024| 1550000.0|        -5804.25903|
|  98112|  490000.0|      -5823.8757792|
|  98108|  442000.0|      -5815.9125122|
|  98023|  405000.0|      -5791.1905024|
|  98006| 1355000.0|      -5807.5449804|
|  98031|  255000.0|      -5791.8063862|
|  98166|  110000.0| -5808.270063999999|
|  98033|  606000.0|-5827.2604200000005|
|  98027|  658588.0| -5796.512861499999|
|  98055|  238000.0|      -5796.0984288|
|  98059|  439950.0|      -5800.2887712|
|  98077|  522250.0|      -5828.7616882|
|  98107|  430000.0|       -5834.380256|
|  98107|  513000.0|       -5833.243293|
|  98177|  580000.0|      -5846.8616672|
|  98146|  750000.0|      -5811.8221164|
+-------+----------+-------------------+
only showing top

--------------------

### Agrupamientos 

In [16]:
#Se agrupa por numero de habitaciones, baños y precio promedio por zipcode
house.groupBy(['zipcode','bedrooms','bathrooms']).avg('Price').show()

+-------+--------+---------+------------------+
|zipcode|bedrooms|bathrooms|        avg(Price)|
+-------+--------+---------+------------------+
|  98023|     4.0|      2.5| 327764.6551724138|
|  98117|     3.0|      3.5|          608250.0|
|  98072|     3.0|      2.0|          505125.0|
|  98144|     3.0|      2.5|          542555.0|
|  98023|     5.0|     1.75|          318250.0|
|  98056|     4.0|      3.5| 637983.3333333334|
|  98056|     2.0|      1.0|239543.16666666666|
|  98133|     4.0|     2.75|         441686.25|
|  98126|     4.0|      2.5|          567500.0|
|  98052|     4.0|      2.0| 617985.7142857143|
|  98011|     2.0|      1.0|          323875.0|
|  98125|     4.0|      4.5|          500000.0|
|  98006|     5.0|      2.0|          585000.0|
|  98133|     2.0|     0.75|          230000.0|
|  98039|     4.0|     2.25|         1515000.0|
|  98039|     3.0|      1.0|          917500.0|
|  98056|     5.0|      3.5| 557685.7142857143|
|  98006|     4.0|      2.0| 499383.3333